# Đề 7 — v5: chẩn đoán SAM 3.1 (dump track + nối ID qua chunk) và baseline đủ 9 video AIC21
Kết quả gộp: out/de7_v5_results.json

In [ ]:
import os, glob, json, subprocess, zipfile, shutil, time, random, csv
T_START=time.time(); os.chdir('/kaggle/working'); os.makedirs('src', exist_ok=True); os.makedirs('configs', exist_ok=True); os.makedirs('out', exist_ok=True)
open('src/pipeline.py','w').write('"""De 7 - phat hien & canh bao xe dung/do trai phep.\nBackend: \'yolo\' (baseline: YOLO26 + ByteTrack/BoT-SORT) | \'sam3\' (loi: SAM 3.1 multiplex video, text prompt).\nRe-ID: DINOv3 (tuy chon) noi lai timer dwell khi track mat/doi ID (che khuat, ranh gioi chunk).\nLogic: ROI polygon -> dung yen -> dwell timer -> phan biet ket xe -> alert CSV/JSON + snapshot + clip.\n"""\nimport argparse, collections, csv, json, os, shutil, sys, tempfile, time\nimport cv2, numpy as np\n\nVEH = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}  # COCO ids\n\n# ---------------- backends: yield (frame_idx, frame_bgr, dets[{id,box,cls,mask}]) ----------------\nMAX_SIDE = 0  # >0: thu nho frame (canh dai) truoc moi xu ly -> giam VRAM/IO cho SAM 3.1\n\ndef frames(video, stride, max_frames):\n    cap = cv2.VideoCapture(video); fi = -1\n    while True:\n        ok, f = cap.read(); fi += 1\n        if not ok or (max_frames and fi >= max_frames): break\n        if fi % stride == 0:\n            if MAX_SIDE and max(f.shape[:2]) > MAX_SIDE:\n                r = MAX_SIDE / max(f.shape[:2]); f = cv2.resize(f, (round(f.shape[1] * r), round(f.shape[0] * r)), interpolation=cv2.INTER_AREA)\n            yield fi, f\n\ndef yolo_backend(a, timers):\n    from ultralytics import YOLO\n    model = YOLO(a.model)\n    n = 0\n    for fi, f in frames(a.video, a.stride, a.max_frames):\n        t0 = time.perf_counter()\n        ck = n // a.yolo_reset if a.yolo_reset else 0  # gia lap ranh gioi chunk: reset tracker -> ID moi\n        r = model.track(f, persist=not (a.yolo_reset and n % a.yolo_reset == 0 and n > 0),  tracker=a.tracker, imgsz=a.imgsz, conf=a.conf,\n                        classes=list(VEH), device=a.device, verbose=False, half=a.half)[0]\n        timers["model"] += time.perf_counter() - t0\n        dets = []\n        if r.boxes.id is not None:\n            for b, tid, c, sc in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().cpu().tolist(), r.boxes.cls.int().cpu().tolist(), r.boxes.conf.cpu().tolist()):\n                dets.append({"id": ck * 100000 + tid, "box": b, "cls": VEH.get(c, str(c)), "mask": None, "score": float(sc)})\n        n += 1\n        yield fi, f, dets\n\ndef sam3_backend(a, timers):\n    """SAM 3.1: chia video thanh chunk (frame da lay mau -> thu muc JPEG), moi chunk 1 session + text prompt.\n    ID giua cac chunk duoc noi bang IoU + DINOv3 trong DwellEngine (offset ID theo chunk)."""\n    import torch\n    from sam3.model_builder import build_sam3_multiplex_video_predictor\n    # T4 (sm75): sam3 decoder ep SDPA FLASH (khong ho tro bf16 tren Turing) -> cho phep EFFICIENT/MATH\n    import sam3.model.decoder as _dec\n    from torch.nn.attention import sdpa_kernel as _sk, SDPBackend as _B\n    _dec.sdpa_kernel = lambda *x, **y: _sk([_B.FLASH_ATTENTION, _B.EFFICIENT_ATTENTION, _B.MATH])\n    # bf16 khong co kernel EFFICIENT tren sm75 -> roi ve MATH (attention matrix ~5 GB, OOM). Ep q,k,v sang fp16 cho attention.\n    import types, torch.nn.functional as _F\n    _bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8\n    def _sdpa(q, k, v, *x, **y):\n        dt = q.dtype\n        if not _bf16_ok and dt == torch.bfloat16: q, k, v = q.half(), k.half(), v.half()\n        # chia query thanh block de neu roi ve MATH thi ma tran attention nho (<~0.5 GB)\n        nq, nk = q.shape[-2], k.shape[-2]\n        bs = max(256, int(1.2e8 // max(nk * max(q[..., :1, :1].numel(), 1), 1)))\n        if nq * nk * q[..., :1, :1].numel() > 2e8 and not y.get("attn_mask") is not None:\n            out = torch.cat([_F.scaled_dot_product_attention(q[..., i:i + bs, :], k, v, *x, **y) for i in range(0, nq, bs)], dim=-2)\n        else:\n            out = _F.scaled_dot_product_attention(q, k, v, *x, **y)\n        return out.to(dt)\n    _dec.torchF = types.SimpleNamespace(**{n: getattr(_F, n) for n in dir(_F) if not n.startswith("__")})\n    _dec.torchF.scaled_dot_product_attention = _sdpa\n    kw = dict(use_fa3=a.fa3, compile=False, max_num_objects=a.max_objects)\n    if a.sam3_ckpt: kw["checkpoint_path"] = a.sam3_ckpt\n    pred = build_sam3_multiplex_video_predictor(**kw)\n    # workaround upstream (sam3@7f473cf): base start_session truyen offload_state_to_cpu nhung multiplex init_state khong nhan\n    _orig = pred.model.init_state\n    def _init_state(**k):\n        k.pop("offload_state_to_cpu", None); return _orig(**k)\n    pred.model.init_state = _init_state\n    # T4 16GB: detector mac dinh batch 16 frame 1008x1008 -> OOM; giam batch grounding/postprocess\n    for attr in ("batched_grounding_batch_size", "postprocess_batch_size"):\n        if hasattr(pred.model, attr): setattr(pred.model, attr, a.det_batch)\n    print("SAM3 loaded, VRAM MB", round(torch.cuda.memory_allocated() / 2**20), flush=True)\n    buf, chunk_id = [], 0\n    def run_chunk(buf, chunk_id):\n        tmp = tempfile.mkdtemp(prefix="sam3chunk_")\n        for i, (fi, f) in enumerate(buf): cv2.imwrite(f"{tmp}/{i:05d}.jpg", f, [cv2.IMWRITE_JPEG_QUALITY, 95])\n        H, W = buf[0][1].shape[:2]\n        t0 = time.perf_counter()\n        with torch.autocast("cuda", dtype=a.amp_dtype, enabled=a.amp_dtype is not None):\n            sid = pred.handle_request(dict(type="start_session", resource_path=tmp, offload_video_to_cpu=a.offload_video))["session_id"]\n            pred.handle_request(dict(type="add_prompt", session_id=sid, frame_index=0, text=a.prompt))\n            outs = {r["frame_index"]: r["outputs"] for r in pred.handle_stream_request(dict(type="propagate_in_video", session_id=sid))}\n            pred.handle_request(dict(type="close_session", session_id=sid))\n        timers["model"] += time.perf_counter() - t0\n        shutil.rmtree(tmp, ignore_errors=True)\n        for i, (fi, f) in enumerate(buf):\n            o = outs.get(i, {}); dets = []\n            ids = o.get("out_obj_ids", np.zeros(0)); bx = o.get("out_boxes_xywh", np.zeros((0, 4))); ms = o.get("out_binary_masks")\n            pr = o.get("out_probs", None)\n            for k, oid in enumerate(np.asarray(ids).tolist()):\n                x, y, w, h = [float(v) for v in bx[k]]\n                if max(x, y, w, h) <= 1.5: x, y, w, h = x * W, y * H, w * W, h * H\n                if w < 2 or h < 2: continue\n                dets.append({"id": chunk_id * 100000 + int(oid), "box": np.array([x, y, x + w, y + h]), "cls": a.prompt,\n                             "mask": ms[k] if ms is not None else None, "score": float(np.asarray(pr)[k]) if pr is not None and len(pr) > k else None})\n            yield fi, f, dets\n    for fi, f in frames(a.video, a.stride, a.max_frames):\n        buf.append((fi, f))\n        if len(buf) >= a.chunk_frames:\n            yield from run_chunk(buf, chunk_id); buf, chunk_id = [], chunk_id + 1\n            torch.cuda.empty_cache()\n    if buf: yield from run_chunk(buf, chunk_id)\n\n# ---------------- DINOv3 re-ID ----------------\nclass Reid:\n    def __init__(s, name, device):\n        import torch\n        from transformers import AutoImageProcessor, AutoModel\n        s.torch, s.dev = torch, device\n        s.proc = AutoImageProcessor.from_pretrained(name); s.m = AutoModel.from_pretrained(name).to(device).eval()\n    def embed(s, frame, box, mask=None):\n        x1, y1, x2, y2 = [int(v) for v in box]; H, W = frame.shape[:2]\n        x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)\n        crop = frame[y1:y2, x1:x2].copy()\n        if crop.size == 0: return None\n        if mask is not None and mask.shape[:2] == (H, W):\n            crop[~mask[y1:y2, x1:x2].astype(bool)] = 0\n        inp = s.proc(images=cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), return_tensors="pt").to(s.dev)\n        with s.torch.no_grad():\n            e = s.m(**inp).pooler_output[0]\n        e = e / e.norm(); return e.float().cpu().numpy()\n\nclass Stitcher:\n    """Noi ID qua ranh gioi chunk (SAM 3.1 moi chunk 1 session; YOLO reset tracker): ID cuc bo moi xuat hien\n    -> ghep voi track toan cuc vua mat (khong co mat o frame nay, mat <= gap_s) neu IoU >= iou_thr\n    va (neu co DINOv3) cosine >= sim_thr. Khong khop -> ID toan cuc moi."""\n    def __init__(s, reid, iou_thr=0.3, sim_thr=0.5, gap_s=20.0, emb_every=10):\n        s.reid, s.iou_thr, s.sim_thr, s.gap_s, s.emb_every = reid, iou_thr, sim_thr, gap_s, emb_every\n        s.map, s.g, s.next, s.n_stitch, s.time = {}, {}, 1, 0, 0.0\n    def _emb(s, frame, d):\n        if s.reid is None: return None\n        t0 = time.perf_counter(); e = s.reid.embed(frame, d["box"], d.get("mask")); s.time += time.perf_counter() - t0; return e\n    def __call__(s, frame, t, dets):\n        used = {s.map[d["id"]] for d in dets if d["id"] in s.map}\n        for d in dets:\n            lid = d["id"]; d["raw_id"] = lid\n            if lid not in s.map:\n                cand = [(iou(g["box"], d["box"]), gid) for gid, g in s.g.items() if gid not in used and 0 < t - g["t"] <= s.gap_s]\n                cand = sorted([c for c in cand if c[0] >= s.iou_thr], reverse=True)\n                e = self_e = s._emb(frame, d) if cand else None\n                gid = None\n                for _, cg in cand:\n                    ge = s.g[cg]["emb"]\n                    if e is None or ge is None or float(np.dot(e, ge)) >= s.sim_thr: gid = cg; break\n                if gid is None: gid = s.next; s.next += 1\n                else: s.n_stitch += 1\n                s.map[lid] = gid; used.add(gid)\n                s.g.setdefault(gid, {"emb": None, "n": 0})\n                if self_e is not None: s.g[gid]["emb"] = self_e\n            gid = s.map[lid]; g = s.g[gid]; g["box"], g["t"] = d["box"], t; g["n"] += 1\n            if s.reid is not None and (g["emb"] is None or g["n"] % s.emb_every == 0): g["emb"] = s._emb(frame, d)\n            d["id"] = gid\n        for gid in [k for k, v in s.g.items() if t - v.get("t", t) > s.gap_s * 3]: del s.g[gid]\n        return dets\n\ndef iou(a, b):\n    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))\n    i = ix * iy; u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i\n    return i / u if u > 0 else 0\n\n# ---------------- logic ----------------\ndef load_zones(path, w, h):\n    cfg = json.load(open(path)); zones = []\n    for z in cfg["zones"]:\n        pts = np.array(z["polygon"], dtype=np.float32)\n        if pts.max() <= 1.0: pts *= [w, h]\n        zones.append({"name": z["name"], "poly": pts.astype(np.int32), "max_dwell_s": float(z.get("max_dwell_s", cfg.get("max_dwell_s", 10)))})\n    return cfg, zones\n\ndef in_zone(zones, x, y):\n    for z in zones:\n        if cv2.pointPolygonTest(z["poly"], (float(x), float(y)), False) >= 0: return z\n    return None\n\nclass Track:\n    def __init__(s, tid, cls):\n        s.id, s.cls = tid, cls; s.hist = collections.deque(maxlen=256)\n        s.still_since = None; s.last_seen = 0; s.alerted = False; s.zone = None; s.emb = None; s.is_still = False; s.box = None\n    def stationary(s, t, win_s, tol):\n        old = [p for p in s.hist if t - p[0] >= win_s]\n        if not old: return False\n        p0, p1 = old[-1], s.hist[-1]\n        return np.hypot(p1[1]-p0[1], p1[2]-p0[2]) < tol * max(p1[3], 1)\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--video", required=True); ap.add_argument("--zones", required=True)\n    ap.add_argument("--backend", default="yolo", choices=["yolo", "sam3"])\n    ap.add_argument("--model", default="yolo26n.pt"); ap.add_argument("--tracker", default="bytetrack.yaml")\n    ap.add_argument("--imgsz", type=int, default=640); ap.add_argument("--conf", type=float, default=0.25)\n    ap.add_argument("--device", default=None); ap.add_argument("--half", action="store_true")\n    ap.add_argument("--prompt", default="car"); ap.add_argument("--sam3_ckpt", default=None)\n    ap.add_argument("--chunk_frames", type=int, default=150); ap.add_argument("--max_objects", type=int, default=16)\n    ap.add_argument("--fa3", action="store_true"); ap.add_argument("--offload_video", action="store_true"); ap.add_argument("--det_batch", type=int, default=16); ap.add_argument("--amp", default="fp16", choices=["none", "fp16", "bf16"])\n    ap.add_argument("--reid", default="none", help="none | HF id, vd facebook/dinov3-vits16-pretrain-lvd1689m")\n    ap.add_argument("--reid_thr", type=float, default=0.75); ap.add_argument("--dedup_iou", type=float, default=0.3); ap.add_argument("--min_area", type=float, default=200)\n    ap.add_argument("--stride", type=int, default=1); ap.add_argument("--max_frames", type=int, default=0)\n    ap.add_argument("--max_side", type=int, default=0, help="thu nho canh dai frame (px), 0 = giu nguyen")\n    ap.add_argument("--dump_tracks", action="store_true", help="ghi out/tracks.csv: frame,t,raw_id,id,x1,y1,x2,y2,score,still,dwell")\n    ap.add_argument("--stitch", action="store_true", help="noi ID qua ranh gioi chunk (IoU + DINOv3 neu co --reid)")\n    ap.add_argument("--stitch_iou", type=float, default=0.3); ap.add_argument("--stitch_sim", type=float, default=0.5); ap.add_argument("--stitch_gap_s", type=float, default=20.0)\n    ap.add_argument("--yolo_reset", type=int, default=0, help="YOLO: reset tracker moi N frame mau (gia lap chunk de test stitch)")\n    ap.add_argument("--sanity", type=int, default=0, help="chi chay N frame dau, in SANITY so box; exit 3 neu 0 box")\n    ap.add_argument("--out", default="outputs/run"); ap.add_argument("--save_video", action="store_true")\n    a = ap.parse_args()\n    global MAX_SIDE; MAX_SIDE = a.max_side\n    import torch; torch.set_num_threads(int(os.environ.get("TORCH_THREADS", "4")))\n    cuda = torch.cuda.is_available() and a.device != "cpu"\n    a.amp_dtype = None if (a.amp == "none" or not cuda) else (torch.float16 if a.amp == "fp16" else torch.bfloat16)\n    if cuda: torch.cuda.reset_peak_memory_stats()\n    os.makedirs(a.out + "/evidence", exist_ok=True)\n    cap = cv2.VideoCapture(a.video); fps = cap.get(cv2.CAP_PROP_FPS) or 25; W, H = int(cap.get(3)), int(cap.get(4)); cap.release()\n    if a.max_side and max(W, H) > a.max_side: r = a.max_side / max(W, H); W, H = round(W * r), round(H * r)\n    backend = yolo_backend if a.backend == "yolo" else sam3_backend\n    if a.sanity:  # kiem tra nhanh truoc khi chay full (vd prompt sai -> 0 box)\n        a.chunk_frames = a.sanity; nb, nf = 0, 0\n        for fi, _, dets in backend(a, {"model": 0.0, "reid": 0.0}):\n            nb += len(dets); nf += 1\n            if nf >= a.sanity: break\n        vram = round(torch.cuda.max_memory_allocated() / 2**20, 1) if cuda else None\n        print("SANITY", json.dumps({"video": a.video, "prompt": a.prompt, "frames": nf, "boxes": nb, "vram_peak_mb": vram}), flush=True)\n        sys.exit(0 if nb > 0 else 3)\n    cfg, zones = load_zones(a.zones, W, H)\n    win_s, tol, grace_s = cfg.get("still_window_s", 2.0), cfg.get("still_tol", 0.15), cfg.get("lost_grace_s", 3.0)\n    reid_keep_s = cfg.get("reid_keep_s", 30.0)\n    jam_ratio, jam_min = cfg.get("jam_ratio", 0.6), cfg.get("jam_min_vehicles", 4)\n    ring = collections.deque(maxlen=max(1, int(cfg.get("clip_s", 6) * fps / a.stride)))\n    reid = Reid(a.reid, "cuda" if cuda else "cpu") if a.reid != "none" else None\n    vw = cv2.VideoWriter(a.out + "/annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps / a.stride, (W, H)) if a.save_video else None\n    timers = {"model": 0.0, "reid": 0.0}\n    tracks, lost, alerts, n_proc, jam_frames, n_reid = {}, {}, [], 0, 0, 0\n    backend = yolo_backend if a.backend == "yolo" else sam3_backend\n    stitcher = Stitcher(reid, a.stitch_iou, a.stitch_sim, a.stitch_gap_s) if a.stitch else None\n    dump = None\n    if a.dump_tracks:\n        dump_f = open(a.out + "/tracks.csv", "w", newline=""); dump = csv.writer(dump_f)\n        dump.writerow(["frame", "t", "raw_id", "id", "x1", "y1", "x2", "y2", "score", "still", "dwell"])\n    t_start = time.perf_counter()\n    for fi, frame, dets in backend(a, timers):\n        t = fi / fps; n_proc += 1; seen = []\n        if stitcher: dets = stitcher(frame, t, dets)\n        for d in dets:\n            tr = tracks.get(d["id"])\n            if tr is None:\n                tr = tracks[d["id"]] = Track(d["id"], d["cls"])\n                # re-ID: ke thua timer cua track da mat tai cung vi tri\n                cand = [l for l in lost.values() if iou(l.box, d["box"]) >= 0.5]\n                if cand:\n                    ok = None\n                    if reid is not None:\n                        t0 = time.perf_counter(); e = reid.embed(frame, d["box"], d["mask"]); timers["reid"] += time.perf_counter() - t0\n                        tr.emb = e\n                        sims = [(float(np.dot(e, l.emb)), l) for l in cand if l.emb is not None and e is not None]\n                        if sims and max(sims, key=lambda s: s[0])[0] >= a.reid_thr: ok = max(sims, key=lambda s: s[0])[1]\n                    else:\n                        ok = max(cand, key=lambda l: iou(l.box, d["box"])) if a.backend == "sam3" else None\n                    if ok is not None:\n                        tr.still_since, tr.zone, tr.alerted, tr.hist = ok.still_since, ok.zone, ok.alerted, ok.hist\n                        del lost[ok.id]; n_reid += 1\n            x1, y1, x2, y2 = d["box"]\n            tr.hist.append((t, (x1+x2)/2, y2, y2-y1)); tr.last_seen = t; tr.box = tuple(int(v) for v in d["box"]); tr.mask = d["mask"]\n            seen.append(tr)\n        for tr in seen:\n            st = tr.stationary(t, win_s, tol); tr.is_still = st\n            z = in_zone(zones, tr.hist[-1][1], tr.hist[-1][2])\n            if st and z is not None:\n                if tr.still_since is None or tr.zone is not z: tr.still_since, tr.zone = t - win_s, z\n                if reid is not None and tr.emb is None:\n                    t0 = time.perf_counter(); tr.emb = reid.embed(frame, tr.box, tr.mask); timers["reid"] += time.perf_counter() - t0\n            elif not st:\n                tr.still_since, tr.zone, tr.alerted = None, None, False\n        if dump:\n            for tr in seen:\n                d0 = next((d for d in dets if d["id"] == tr.id), {})\n                dump.writerow([fi, round(t, 2), d0.get("raw_id", tr.id), tr.id, *tr.box, None if d0.get("score") is None else round(d0["score"], 3),\n                               int(tr.is_still), round(t - tr.still_since, 1) if tr.still_since is not None else 0])\n        n_still = sum(tr.is_still for tr in seen)\n        jam = bool(len(seen) >= jam_min and n_still / max(len(seen), 1) >= jam_ratio); jam_frames += jam\n        for tid in [k for k, v in tracks.items() if t - v.last_seen > 0 and v not in seen]:\n            v = tracks[tid]\n            if v.still_since is not None: lost[tid] = v\n            if t - v.last_seen > grace_s: del tracks[tid]\n        for k in [k for k, v in lost.items() if t - v.last_seen > reid_keep_s]: del lost[k]\n        vis = frame.copy()\n        for z in zones: cv2.polylines(vis, [z["poly"]], True, (0, 0, 255), 2)\n        for tr in seen:\n            dwell = (t - tr.still_since) if tr.still_since is not None else 0\n            x1, y1, x2, y2 = tr.box\n            dup = any(iou(e["bbox"], tr.box) >= a.dedup_iou for e in alerts) if a.dedup_iou > 0 else False\n            small = (x2 - x1) * (y2 - y1) < a.min_area\n            if tr.zone and dwell >= tr.zone["max_dwell_s"] and not tr.alerted and (dup or small) and not jam:\n                tr.alerted = True  # cung vi tri voi canh bao truoc (cung su co) hoac xe qua xa -> khong canh bao lai\n            if tr.zone and dwell >= tr.zone["max_dwell_s"] and not tr.alerted and not jam:\n                tr.alerted = True\n                ev = {"video": os.path.basename(a.video), "backend": a.backend, "track_id": tr.id, "class": tr.cls, "zone": tr.zone["name"],\n                      "stop_start_s": round(tr.still_since, 2), "alert_time_s": round(t, 2), "dwell_s": round(dwell, 2), "bbox": list(tr.box), "frame": fi}\n                im = vis.copy(); cv2.rectangle(im, (x1, y1), (x2, y2), (0, 0, 255), 3)\n                ev["snapshot"] = f"{a.out}/evidence/alert_{tr.id}_{fi}.jpg"; cv2.imwrite(ev["snapshot"], im)\n                ev["clip"] = f"{a.out}/evidence/alert_{tr.id}_{fi}.mp4"\n                cw = cv2.VideoWriter(ev["clip"], cv2.VideoWriter_fourcc(*"mp4v"), fps / a.stride, (W, H))\n                for f in ring: cw.write(f)\n                cw.write(im); cw.release()\n                alerts.append(ev); print("ALERT", json.dumps(ev, ensure_ascii=False), flush=True)\n            col = (0, 0, 255) if tr.alerted else (0, 200, 255) if tr.still_since is not None else (0, 255, 0)\n            if tr.mask is not None and tr.mask.shape[:2] == vis.shape[:2]:\n                vis[tr.mask.astype(bool)] = (0.6 * vis[tr.mask.astype(bool)] + 0.4 * np.array(col)).astype(np.uint8)\n            cv2.rectangle(vis, (x1, y1), (x2, y2), col, 2); cv2.putText(vis, f"{tr.id % 100000} {dwell:.0f}s", (x1, y1 - 4), 0, 0.5, col, 1)\n        if jam: cv2.putText(vis, "TRAFFIC JAM - alerts suppressed", (10, 30), 0, 0.8, (0, 0, 255), 2)\n        ring.append(vis)\n        if vw: vw.write(vis)\n    wall = time.perf_counter() - t_start\n    if dump: dump_f.close()\n    if stitcher: timers["reid"] += stitcher.time\n    if vw: vw.release()\n    metrics = {"video": a.video, "backend": a.backend, "model": a.model if a.backend == "yolo" else "sam3.1_multiplex", "tracker": a.tracker if a.backend == "yolo" else "sam3.1",\n               "reid": a.reid, "device": torch.cuda.get_device_name(0) if cuda else "cpu", "amp": a.amp if cuda else "none", "stride": a.stride,\n               "frames_processed": n_proc, "fps_model": round(n_proc / max(timers["model"], 1e-9), 2), "reid_time_s": round(timers["reid"], 2),\n               "fps_end2end": round(n_proc / max(wall, 1e-9), 2), "wall_s": round(wall, 1), "n_alerts": len(alerts), "jam_frames": int(jam_frames), "reid_merges": n_reid, "stitches": stitcher.n_stitch if stitcher else 0, "n_ids_stitched": (stitcher.next - 1) if stitcher else None,\n               "vram_peak_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if cuda else None}\n    json.dump(alerts, open(a.out + "/alerts.json", "w"), indent=2, ensure_ascii=False)\n    with open(a.out + "/alerts.csv", "w", newline="") as f:\n        wr = csv.DictWriter(f, fieldnames=["video", "backend", "track_id", "class", "zone", "stop_start_s", "alert_time_s", "dwell_s", "bbox", "frame", "snapshot", "clip"])\n        wr.writeheader(); [wr.writerow(e) for e in alerts]\n    json.dump(metrics, open(a.out + "/metrics.json", "w"), indent=2)\n    print("METRICS", json.dumps(metrics), flush=True)\n\nif __name__ == "__main__":\n    main()\n')
open('src/evaluate.py','w').write('"""Danh gia muc su kien.\nGT json [{"video","zone","stop_start_s","stop_end_s"}]; pred = alerts.json cua pipeline.\n--dedup_iou: gop cac canh bao cung video co bbox IoU >= nguong (cung mot xe/su co dung yen, ID bi doi) -> giu canh bao som nhat.\n--min_area: bo canh bao co bbox nho hon (px^2) (xe qua xa).\nTP neu cung video+zone va alert_time trong [stop_start, stop_end + tol]. Bao cao P/R/F1, MAE thoi diem bat dau dung, do tre canh bao."""\nimport argparse, json, numpy as np\n\ndef iou(a, b):\n    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = ix * iy\n    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i\n    return i / u if u > 0 else 0\n\ndef filter_alerts(pr, dedup_iou=0.0, min_area=0):\n    pr = [p for p in pr if (p["bbox"][2]-p["bbox"][0]) * (p["bbox"][3]-p["bbox"][1]) >= min_area]\n    if dedup_iou <= 0: return pr\n    keep = []\n    for p in sorted(pr, key=lambda e: e["alert_time_s"]):\n        if not any(k["video"] == p["video"] and iou(k["bbox"], p["bbox"]) >= dedup_iou for k in keep): keep.append(p)\n    return keep\n\ndef score(gt, pr, tol=30.0):\n    used, tp, errs, lat = set(), 0, [], []\n    for p in sorted(pr, key=lambda e: e["alert_time_s"]):\n        for i, g in enumerate(gt):\n            if i in used or g["video"] != p["video"] or g["zone"] != p["zone"]: continue\n            if g["stop_start_s"] <= p["alert_time_s"] <= g["stop_end_s"] + tol:\n                used.add(i); tp += 1; errs.append(abs(p["stop_start_s"] - g["stop_start_s"])); lat.append(p["alert_time_s"] - g["stop_start_s"]); break\n    P = tp / max(len(pr), 1); R = tp / max(len(gt), 1)\n    return {"tp": tp, "fp": len(pr) - tp, "fn": len(gt) - tp, "precision": round(P, 3), "recall": round(R, 3),\n            "f1": round(2*P*R/max(P+R, 1e-9), 3), "start_mae_s": round(float(np.mean(errs)), 2) if errs else None,\n            "mean_alert_latency_s": round(float(np.mean(lat)), 2) if lat else None}\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument("gt"); ap.add_argument("pred"); ap.add_argument("tol", nargs="?", type=float, default=30)\n    ap.add_argument("--dedup_iou", type=float, default=0.0); ap.add_argument("--min_area", type=float, default=0)\n    a = ap.parse_args()\n    gt, pr = json.load(open(a.gt)), json.load(open(a.pred))\n    print(json.dumps(score(gt, filter_alerts(pr, a.dedup_iou, a.min_area), a.tol)))\n')
open('configs/zones_aic_full.json','w').write('{"still_window_s": 3.0, "still_tol": 0.1, "lost_grace_s": 4.0, "reid_keep_s": 60, "jam_ratio": 0.6, "jam_min_vehicles": 4, "clip_s": 8,\n "zones": [{"name": "all", "polygon": [[0,0],[1,0],[1,1],[0,1]], "max_dwell_s": 20}]}\n')


In [ ]:
!pip -q install gdown "git+https://github.com/facebookresearch/sam3.git@7f473cf" ultralytics lap "transformers>=4.56" einops decord pycocotools 2>&1 | tail -1
WDIR = os.path.dirname(glob.glob('/kaggle/input/**/sam31_sha256.txt', recursive=True)[0])
os.makedirs('weights', exist_ok=True); SAM3_CKPT='weights/sam3.1_multiplex.pt'
if not os.path.exists(SAM3_CKPT): subprocess.run(f'cat {WDIR}/sam31_part_* > {SAM3_CKPT}', shell=True, check=True)
r='dinov3-vits16-pretrain-lvd1689m'; os.makedirs(f'weights/{r}', exist_ok=True)
for f in glob.glob(f'{WDIR}/{r}__*'):
    dst=f'weights/{r}/'+os.path.basename(f).split('__',1)[1]
    if not os.path.exists(dst): os.symlink(f,dst)
DINO=f'weights/{r}'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
Z='/tmp/aic21.zip'
if not os.path.exists(Z): subprocess.run(f'gdown --fuzzy "https://drive.google.com/file/d/1gsR4pIEiWK7htPmIjiZjXetaa1X-T-Am/view" -O {Z}', shell=True)
zf = zipfile.ZipFile(Z); names = zf.namelist()
gtf = [n for n in names if n.endswith('train-anomaly-results.txt')][0]
rows = [l.split() for l in zf.read(gtf).decode().splitlines() if len(l.split()) >= 3 and l.split()[0].isdigit()]
vids = {n.split('/')[-1]: n for n in names if n.endswith('.mp4') and 'train' in n.lower()}
# cung tap 9 video nhu v2 (make_nb_aic.py): 6 duong tinh dau + 3 am tinh random.seed(0)
pos = sorted({int(r[0]) for r in rows}); neg = [i for i in range(1, 101) if i not in pos and f'{i}.mp4' in vids]
random.seed(0); sel_pos = pos[:6]; sel_neg = random.sample(neg, 3); SEL = sel_pos + sel_neg; print('SEL', SEL)
os.makedirs('aic', exist_ok=True)
for i in SEL:
    with zf.open(vids[f'{i}.mp4']) as s, open(f'aic/{i}.mp4','wb') as d: shutil.copyfileobj(s, d)
GT = [{'video': f'{int(r[0])}.mp4', 'zone': 'all', 'stop_start_s': float(r[1]), 'stop_end_s': float(r[2])} for r in rows if int(r[0]) in SEL]
json.dump(GT, open('aic/gt.json','w'), indent=1); print(GT); os.remove(Z)

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
import cv2
BUDGET_S = float(os.environ.get('BUDGET_S', '6300'))
RES = {'settings': {'gpu': torch.cuda.get_device_name(0), 'videos': SEL, 'gt': GT}, 'sam_diag': {}, 'aic9': {}}
def save(): json.dump(RES, open('out/de7_v5_results.json','w'), indent=1)
def run(cmd, tmo):
    try: r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=tmo); return r
    except subprocess.TimeoutExpired: return None
def track_stats(p):
    R = list(csv.DictReader(open(p))) if os.path.exists(p) else []
    if not R: return {'rows': 0}
    fr = sorted({int(x['frame']) for x in R})
    return {'rows': len(R), 'frames_with_dets': len(fr), 'raw_ids': len({x['raw_id'] for x in R}), 'ids': len({x['id'] for x in R}),
            'still_rows': sum(x['still'] == '1' for x in R), 'max_dwell_s': max(float(x['dwell']) for x in R),
            'mean_dets_per_frame': round(len(R) / max(len(fr), 1), 2)}
# ---- A) SAM 3.1: video 35 (su co 106-185 s), 240 s dau, 1 fps, chunk 60; khong stitch vs stitch IoU+DINOv3 ----
V = 35; fps = cv2.VideoCapture(f'aic/{V}.mp4').get(cv2.CAP_PROP_FPS) or 30
SAMA = (f'--video aic/{V}.mp4 --stride {round(fps)} --max_frames {int(240*fps)} --max_side 640 --zones configs/zones_aic_full.json '
        f'--backend sam3 --sam3_ckpt {SAM3_CKPT} --amp fp16 --prompt car --offload_video --chunk_frames 60 --max_objects 8 --det_batch 1 --dump_tracks')
RES['settings']['sam'] = SAMA
r = run(f'python src/pipeline.py {SAMA} --sanity 5 --out out/sanity', 900)
RES['sanity'] = [l for l in (r.stdout if r else '').splitlines() if l.startswith('SANITY')] or (r.stderr[-1500:] if r else 'timeout'); save(); print(RES['sanity'])
gt35 = [g for g in GT if g['video'] == f'{V}.mp4' and g['stop_start_s'] < 220]
for g in gt35: g['stop_end_s'] = min(g['stop_end_s'], 240)
json.dump(gt35, open('out/gt35.json','w'))
for name, extra in [('sam31_nostitch', ''), ('sam31_stitch_dino', f'--stitch --reid {DINO}')]:
    left = BUDGET_S - (time.time() - T_START)
    if left < 300: print('BUDGET HIT', name); continue
    r = run(f'python src/pipeline.py {SAMA} {extra} --out out/{name}', min(left - 60, 2400))
    m = json.load(open(f'out/{name}/metrics.json')) if os.path.exists(f'out/{name}/metrics.json') else {'error': (r.stderr[-1500:] if r else 'timeout')}
    ev = json.loads(subprocess.run(f'python src/evaluate.py out/gt35.json out/{name}/alerts.json 30 --dedup_iou 0.3', shell=True, capture_output=True, text=True).stdout or '{}') if 'error' not in m else {}
    RES['sam_diag'][name] = {'metrics': m, 'eval_dedup0.3': ev, 'tracks': track_stats(f'out/{name}/tracks.csv')}; save()
    print(name, json.dumps(RES['sam_diag'][name]), flush=True)
shutil.rmtree('weights', ignore_errors=True)

In [ ]:
# ---- B) 9 video AIC21, 2 fps (stride 15), ca khung hinh, dwell 20 s, nhu v2; eval tho va dedup IoU 0.3 ----
CFG = {'yolo26s_bytetrack': '--backend yolo --model yolo26s.pt', 'yolo26s_bytetrack_dino': f'--backend yolo --model yolo26s.pt --reid {DINO}'}
if not os.path.exists(DINO + '/config.json'):   # weights da xoa o tren -> tao lai symlink DINOv3
    os.makedirs(DINO, exist_ok=True)
    for f in glob.glob(f'{WDIR}/dinov3-vits16-pretrain-lvd1689m__*'): os.symlink(f, f'{DINO}/' + os.path.basename(f).split('__',1)[1])
for name, args in CFG.items():
    ms, al = [], []
    for i in SEL:
        left = BUDGET_S - (time.time() - T_START)
        if left < 120: print('BUDGET HIT', name, i); continue
        r = run(f'python src/pipeline.py --video aic/{i}.mp4 --zones configs/zones_aic_full.json --stride 15 --out out/{name}/{i} {args}', left)
        if os.path.exists(f'out/{name}/{i}/metrics.json'): ms.append(json.load(open(f'out/{name}/{i}/metrics.json'))); al += json.load(open(f'out/{name}/{i}/alerts.json'))
        else: print(name, i, (r.stderr[-800:] if r else 'timeout'))
    done = [os.path.basename(m['video']) for m in ms]; os.makedirs(f'out/{name}', exist_ok=True)
    json.dump(al, open(f'out/{name}/all_alerts.json','w')); json.dump([g for g in GT if g['video'] in done], open(f'out/{name}/gt.json','w'))
    ev = {k: json.loads(subprocess.run(f'python src/evaluate.py out/{name}/gt.json out/{name}/all_alerts.json 30 {a}', shell=True, capture_output=True, text=True).stdout or '{}')
          for k, a in [('raw', ''), ('dedup0.3', '--dedup_iou 0.3')]}
    nf = sum(m['frames_processed'] for m in ms)
    RES['aic9'][name] = {'videos_done': done, 'frames': nf,
        'fps_model': round(nf / max(sum(m['frames_processed'] / max(m['fps_model'], 1e-9) for m in ms), 1e-9), 2) if ms else None,
        'fps_e2e': round(nf / max(sum(m['wall_s'] for m in ms), 1e-9), 2) if ms else None,
        'vram_peak_mb': max(((m['vram_peak_mb'] or 0) for m in ms), default=None), 'eval': ev}
    save(); print(name, json.dumps(RES['aic9'][name]), flush=True)
RES['total_wall_s'] = round(time.time() - T_START); save()
shutil.rmtree('aic', ignore_errors=True); shutil.rmtree('out/sanity', ignore_errors=True)
for p in glob.glob('out/**/evidence/*.mp4', recursive=True): os.remove(p)
print(open('out/de7_v5_results.json').read()[:5000])